# 🦀 Day 21: Mini-Project — Student Grade Management System

Combining structs, enums, Option, Result, pattern matching, and collections!

---

In [ ]:
use std::collections::HashMap;

#[derive(Debug, Clone, PartialEq)]
enum Grade { A, B, C, D, F }

impl Grade {
    fn from_score(score: f64) -> Option<Grade> {
        match score as u32 {
            90..=100 => Some(Grade::A),
            80..=89  => Some(Grade::B),
            70..=79  => Some(Grade::C),
            60..=69  => Some(Grade::D),
            0..=59   => Some(Grade::F),
            _        => None,
        }
    }

    fn gpa_points(&self) -> f64 {
        match self {
            Grade::A => 4.0,
            Grade::B => 3.0,
            Grade::C => 2.0,
            Grade::D => 1.0,
            Grade::F => 0.0,
        }
    }
}

println!("Grade system ready! ✅");

In [ ]:
#[derive(Debug, Clone)]
enum StudentStatus { Active, Graduated, Suspended, OnLeave }

#[derive(Debug, Clone)]
struct Student {
    id: u32,
    name: String,
    scores: HashMap<String, f64>,  // subject → score
    status: StudentStatus,
}

impl Student {
    fn new(id: u32, name: &str) -> Self {
        Student {
            id, name: name.into(),
            scores: HashMap::new(),
            status: StudentStatus::Active,
        }
    }

    fn add_score(&mut self, subject: &str, score: f64) -> Result<(), String> {
        if score < 0.0 || score > 100.0 {
            return Err(format!("Invalid score {}: must be 0-100", score));
        }
        self.scores.insert(subject.to_string(), score);
        Ok(())
    }

    fn average(&self) -> Option<f64> {
        if self.scores.is_empty() { return None; }
        let sum: f64 = self.scores.values().sum();
        Some(sum / self.scores.len() as f64)
    }

    fn letter_grade(&self) -> Option<Grade> {
        self.average().and_then(Grade::from_score)
    }

    fn gpa(&self) -> Option<f64> {
        if self.scores.is_empty() { return None; }
        let total: f64 = self.scores.values()
            .filter_map(|&s| Grade::from_score(s))
            .map(|g| g.gpa_points())
            .sum();
        Some(total / self.scores.len() as f64)
    }

    fn highest_subject(&self) -> Option<(&str, f64)> {
        self.scores.iter()
            .max_by(|a, b| a.1.partial_cmp(b.1).unwrap())
            .map(|(subj, &score)| (subj.as_str(), score))
    }

    fn is_passing(&self) -> bool {
        self.average().map_or(false, |avg| avg >= 60.0)
    }
}

println!("Student struct ready! ✅");

In [ ]:
// Grade Management System
struct GradeSystem {
    students: Vec<Student>,
}

impl GradeSystem {
    fn new() -> Self { GradeSystem { students: Vec::new() } }

    fn add_student(&mut self, student: Student) {
        println!("  ✅ Enrolled: {} (ID: {})", student.name, student.id);
        self.students.push(student);
    }

    fn find_student(&self, id: u32) -> Option<&Student> {
        self.students.iter().find(|s| s.id == id)
    }

    fn find_student_mut(&mut self, id: u32) -> Option<&mut Student> {
        self.students.iter_mut().find(|s| s.id == id)
    }

    fn class_average(&self) -> Option<f64> {
        let avgs: Vec<f64> = self.students.iter()
            .filter_map(|s| s.average())
            .collect();
        if avgs.is_empty() { None }
        else { Some(avgs.iter().sum::<f64>() / avgs.len() as f64) }
    }

    fn top_students(&self, n: usize) -> Vec<&Student> {
        let mut ranked: Vec<&Student> = self.students.iter()
            .filter(|s| s.average().is_some())
            .collect();
        ranked.sort_by(|a, b| {
            b.average().unwrap().partial_cmp(&a.average().unwrap()).unwrap()
        });
        ranked.into_iter().take(n).collect()
    }

    fn report(&self) {
        let border = "═".repeat(70);
        println!("\n╔{}╗", border);
        println!("║{:^70}║", "🎓 STUDENT GRADE REPORT");
        println!("╠{}╣", border);
        println!("║ {:>4} │ {:20} │ {:>6} │ {:>5} │ {:>5} │ {:>7} ║",
            "ID", "Name", "Avg", "Grade", "GPA", "Status");
        println!("╠{}╣", border);

        for s in &self.students {
            let avg = s.average().map_or("-".into(), |a| format!("{:.1}", a));
            let grade = s.letter_grade().map_or("-".into(), |g| format!("{:?}", g));
            let gpa = s.gpa().map_or("-".into(), |g| format!("{:.2}", g));
            let status = if s.is_passing() { "✅ Pass" } else { "❌ Fail" };
            println!("║ {:>4} │ {:20} │ {:>6} │ {:>5} │ {:>5} │ {:>7} ║",
                s.id, s.name, avg, grade, gpa, status);
        }

        println!("╠{}╣", border);
        let class_avg = self.class_average().map_or("-".into(), |a| format!("{:.1}", a));
        println!("║{:^70}║", format!("Class Average: {} │ Total Students: {}", class_avg, self.students.len()));
        println!("╚{}╝", border);
    }
}

println!("GradeSystem ready! ✅");

In [ ]:
// Build the system!
let mut system = GradeSystem::new();

let mut alice = Student::new(1, "Alice Johnson");
alice.add_score("Math", 92.0).ok();
alice.add_score("Science", 88.0).ok();
alice.add_score("English", 95.0).ok();

let mut bob = Student::new(2, "Bob Smith");
bob.add_score("Math", 75.0).ok();
bob.add_score("Science", 68.0).ok();
bob.add_score("English", 72.0).ok();

let mut charlie = Student::new(3, "Charlie Brown");
charlie.add_score("Math", 45.0).ok();
charlie.add_score("Science", 52.0).ok();
charlie.add_score("English", 58.0).ok();

let mut diana = Student::new(4, "Diana Prince");
diana.add_score("Math", 98.0).ok();
diana.add_score("Science", 96.0).ok();
diana.add_score("English", 94.0).ok();

system.add_student(alice);
system.add_student(bob);
system.add_student(charlie);
system.add_student(diana);

system.report();

In [ ]:
// Top students
println!("\n🏆 Top 2 Students:");
for (i, s) in system.top_students(2).iter().enumerate() {
    println!("  #{}: {} (avg: {:.1})", i + 1, s.name, s.average().unwrap());
}

// Find specific student
if let Some(student) = system.find_student(1) {
    println!("\n📋 Student Detail: {}", student.name);
    for (subj, score) in &student.scores {
        let grade = Grade::from_score(*score).unwrap();
        println!("  {} : {:.0} ({:?})", subj, score, grade);
    }
    if let Some((subj, score)) = student.highest_subject() {
        println!("  Best: {} ({:.0})", subj, score);
    }
}

## 🏋️ Extend the System!

In [ ]:
// Challenge 1: Add a method to find students failing a specific subject

// YOUR CODE HERE


In [ ]:
// Challenge 2: Add a method to compute subject-wise class averages
// Returns HashMap<String, f64>

// YOUR CODE HERE


## 🎯 Week 3 Complete! 🎉

You've learned:
- ✅ Structs (regular, tuple, unit)
- ✅ Methods & associated functions
- ✅ Enums with data variants
- ✅ `Option<T>` — null safety
- ✅ `Result<T, E>` — error handling
- ✅ Built a grade management system!

---
**Next Week:** Modules, Crates & Cargo! 📦